In [1]:
"""
GDELT Historical Sentiment Collection for Portfolio
===================================================

Collects weekly sentiment from GDELT database for your portfolio tickers
from 2020 to present. Uses BigQuery for fast processing.

Requirements:
- pip install google-cloud-bigquery pandas
- Google Cloud project with BigQuery enabled
- GDELT BigQuery dataset (free to query up to 1TB/month)

Based on your previous GDELT financial analytics work.
"""

from google.cloud import bigquery
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

class GDELTSentimentCollector:
    """
    Collects historical sentiment from GDELT for portfolio tickers.
    """

    def __init__(self, tickers, start_date='2020-01-01', end_date=None):
        """
        Initialize GDELT sentiment collector.

        Parameters
        ----------
        tickers : list
            List of stock tickers
        start_date : str
            Start date (YYYY-MM-DD)
        end_date : str, optional
            End date (default: today)
        """
        self.tickers = [t.upper() for t in tickers]
        self.start_date = pd.to_datetime(start_date)
        self.end_date = pd.to_datetime(end_date) if end_date else pd.Timestamp.now()

        # Initialize BigQuery client
        try:
            self.client = bigquery.Client()
        except Exception as e:
            print(f"BigQuery client initialization failed: {e}")
            print("Trying without authentication for public datasets...")
            try:
                # Try without authentication for public datasets
                self.client = bigquery.Client(project=None)
            except Exception as e2:
                print(f"Failed to initialize BigQuery client: {e2}")
                print("\nTo fix this, run one of these commands:")
                print("1. gcloud auth application-default login")
                print("2. export GOOGLE_APPLICATION_CREDENTIALS='path/to/your/key.json'")
                raise e2

        # Company name mappings (for better search results)
        self.company_names = {
            'NVDA': 'NVIDIA',
            'AMD': 'AMD Advanced Micro Devices',
            'ASML': 'ASML',
            'MU': 'Micron',
            'AAPL': 'Apple',
            'MSFT': 'Microsoft',
            'GOOG': 'Google Alphabet',
            'RDDT': 'Reddit',
            'SMR': 'NuScale',
            'AI': 'C3.ai',
            'ARBE': 'Arbe Robotics',
            'CHYM': 'Chyma'
        }

        print(f"\nGDELT Sentiment Collector initialized")
        print(f"Date range: {self.start_date.date()} to {self.end_date.date()}")
        print(f"Tickers: {len(self.tickers)}")

    def collect_weekly_sentiment(self, save_path='gdelt_sentiment_weekly.csv'):
        """
        Collect weekly sentiment for all tickers from GDELT.

        Parameters
        ----------
        save_path : str
            Path to save results

        Returns
        -------
        sentiment_df : pd.DataFrame
            Weekly sentiment data
        """
        print(f"\n{'='*70}")
        print("COLLECTING WEEKLY SENTIMENT FROM GDELT")
        print(f"{'='*70}\n")

        all_sentiment = []

        for ticker in self.tickers:
            print(f"[{ticker}] Querying GDELT...")

            # Get sentiment for this ticker
            ticker_sentiment = self._query_gdelt_for_ticker(ticker)

            if ticker_sentiment is not None and not ticker_sentiment.empty:
                # Aggregate to weekly
                weekly_sentiment = self._aggregate_to_weekly(ticker_sentiment, ticker)
                all_sentiment.append(weekly_sentiment)

                print(f"  âœ“ Collected {len(weekly_sentiment)} weeks of data")
            else:
                print(f"  âš ï¸ No data found")

        # Combine all tickers
        if all_sentiment:
            sentiment_df = pd.concat(all_sentiment, ignore_index=True)

            # Pivot to wide format (one row per week, columns for each ticker)
            sentiment_wide = sentiment_df.pivot(
                index='week_end',
                columns='ticker',
                values=['avg_tone', 'article_count']
            )

            # Flatten column names
            sentiment_wide.columns = [f'{ticker}_{metric}' 
                                     for metric, ticker in sentiment_wide.columns]
            sentiment_wide = sentiment_wide.reset_index()

            # Save
            sentiment_df.to_csv(save_path, index=False)
            sentiment_wide.to_csv(save_path.replace('.csv', '_wide.csv'), index=False)

            print(f"\nâœ“ Saved sentiment data to {save_path}")
            print(f"  Long format: {len(sentiment_df)} rows")
            print(f"  Wide format: {len(sentiment_wide)} weeks")

            return sentiment_df, sentiment_wide
        else:
            print("\nNo sentiment data collected")
            return None, None

    def _query_gdelt_for_ticker(self, ticker):
        """
        Query GDELT BigQuery for a single ticker.

        Uses V2TONE (sentiment) from GDELT GKG table.
        """
        company_name = self.company_names.get(ticker, ticker)

        # Build search pattern (ticker OR company name)
        search_pattern = f"({ticker}|{company_name})"

        # GDELT query
        # Note: GDELT 2.0 starts from Feb 2015
        # Using gdelt-bq.gdeltv2.gkg for Global Knowledge Graph
        query = f"""
        SELECT
            DATE(PARSE_TIMESTAMP('%Y%m%d%H%M%S', CAST(DATE AS STRING))) as date,
            V2Tone as tone,
            DocumentIdentifier as url
        FROM
            `gdelt-bq.gdeltv2.gkg`
        WHERE
            DATE >= {self.start_date.strftime('%Y%m%d')}000000
            AND DATE <= {self.end_date.strftime('%Y%m%d')}235959
            AND (
                REGEXP_CONTAINS(V2Themes, r'(?i){search_pattern}')
                OR REGEXP_CONTAINS(V2Persons, r'(?i){search_pattern}')
                OR REGEXP_CONTAINS(V2Organizations, r'(?i){search_pattern}')
                OR REGEXP_CONTAINS(AllNames, r'(?i){search_pattern}')
            )
            AND V2Tone IS NOT NULL
            AND ABS(V2Tone) <= 100  -- Filter extreme outliers
        ORDER BY
            date
        """

        try:
            # Execute query
            query_job = self.client.query(query)
            results = query_job.to_dataframe()

            if not results.empty:
                # Extract tone score (format: "tone,positive,negative,polarity,activity,...")
                results['tone_score'] = results['tone'].str.split(',').str[0].astype(float)
                results = results[['date', 'tone_score', 'url']]

            return results

        except Exception as e:
            print(f"  Error querying GDELT: {e}")
            return None

    def _aggregate_to_weekly(self, daily_data, ticker):
        """
        Aggregate daily sentiment to weekly (Friday close).
        """
        if daily_data is None or daily_data.empty:
            return pd.DataFrame()

        # Set date as index
        daily_data = daily_data.set_index('date')

        # Resample to weekly (Friday)
        weekly = pd.DataFrame({
            'avg_tone': daily_data['tone_score'].resample('W-FRI').mean(),
            'article_count': daily_data['tone_score'].resample('W-FRI').count(),
            'tone_std': daily_data['tone_score'].resample('W-FRI').std()
        })

        # Add ticker column
        weekly['ticker'] = ticker
        weekly['week_end'] = weekly.index

        # Remove weeks with no data
        weekly = weekly[weekly['article_count'] > 0].reset_index(drop=True)

        return weekly

    def create_sentiment_features(self, sentiment_df, lookback_weeks=4):
        """
        Create sentiment features for RL model.

        Parameters
        ----------
        sentiment_df : pd.DataFrame
            Raw weekly sentiment data
        lookback_weeks : int
            Number of weeks for rolling features

        Returns
        -------
        features_df : pd.DataFrame
            Sentiment features
        """
        print(f"\nCreating sentiment features (lookback={lookback_weeks} weeks)...")

        features = []

        for ticker in sentiment_df['ticker'].unique():
            ticker_data = sentiment_df[sentiment_df['ticker'] == ticker].copy()
            ticker_data = ticker_data.sort_values('week_end')

            # Rolling features
            ticker_data['sentiment_ma4w'] = ticker_data['avg_tone'].rolling(4).mean()
            ticker_data['sentiment_ma12w'] = ticker_data['avg_tone'].rolling(12).mean()
            ticker_data['sentiment_std4w'] = ticker_data['avg_tone'].rolling(4).std()
            ticker_data['sentiment_change_1w'] = ticker_data['avg_tone'].diff(1)
            ticker_data['sentiment_change_4w'] = ticker_data['avg_tone'].diff(4)

            # Volume features
            ticker_data['coverage_ma4w'] = ticker_data['article_count'].rolling(4).mean()
            ticker_data['coverage_change_1w'] = ticker_data['article_count'].pct_change(1)

            # Regime features
            ticker_data['sentiment_zscore'] = (
                (ticker_data['avg_tone'] - ticker_data['avg_tone'].rolling(52).mean()) /
                ticker_data['avg_tone'].rolling(52).std()
            )

            features.append(ticker_data)

        features_df = pd.concat(features, ignore_index=True)

        print(f"Created {len(features_df.columns) - 3} sentiment features")

        return features_df



In [2]:


# Your portfolio
PORTFOLIO = ['RDDT', 'NVDA', 'MU', 'AAPL', 'SMR', 'AMD', 'ASML', 
                'MSFT', 'GOOG', 'AI', 'ARBE', 'CHYM']

# Initialize collector
collector = GDELTSentimentCollector(
    tickers=PORTFOLIO,
    start_date='2020-01-01',
    end_date='2025-10-23'
)

# Collect weekly sentiment
sentiment_long, sentiment_wide = collector.collect_weekly_sentiment(
    save_path='data/gdelt_sentiment_weekly.csv'
)

# Create sentiment features
if sentiment_long is not None:
    sentiment_features = collector.create_sentiment_features(sentiment_long)
    sentiment_features.to_csv('data/gdelt_sentiment_features.csv', index=False)

    print(f"\n{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    print(f"Date range: {sentiment_long['week_end'].min()} to {sentiment_long['week_end'].max()}")
    print(f"Total weeks: {sentiment_long['week_end'].nunique()}")
    print(f"Tickers: {sentiment_long['ticker'].nunique()}")
    print(f"\nAverage sentiment by ticker:")
    print(sentiment_long.groupby('ticker')['avg_tone'].agg(['mean', 'std', 'count']))


BigQuery client initialization failed: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.
Trying without authentication for public datasets...
Failed to initialize BigQuery client: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

To fix this, run one of these commands:
1. gcloud auth application-default login
2. export GOOGLE_APPLICATION_CREDENTIALS='path/to/your/key.json'


DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.